In [1]:
import re
from pathlib import Path

import pandas as pd
import pyreadr
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

base = Path(r'C:\Users\leelz\OneDrive\UCSB\Year 4\Quarter 1 (Fall)\PSTAT 197A\module2-claims-team3-1')
claims = pyreadr.read_r(str(base / 'data' / 'claims-clean-example.RData'))['claims_clean']
claims.head()


,neo_search_transaction_id,neo_search_subject_id,original_url,text_tmp,internal_feedback,.id,label,mclass,bclass,text_clean
0,12395162.0,11497914.0,http://hosting-22647.tributes.com/obituary/sho...,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 S...",Possible Fatality,url2183,Possible Fatality,Possible Fatality,Relevant claim content,national obituary search click on the item you...
1,12394582.0,11499214.0,https://www.localcrimenews.com/welcome/arrest/...,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",Potentially unlawful activity,url2136,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the following official arrest record for jonat...
2,12384260.0,11485861.0,https://www.bustedmugshots.com/florida/miami-d...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,N/A: No relevant content.,url1586,N/A: No relevant content.,N/A: No relevant content.,N/A: No relevant content.,did someone you know get arrested in miami dad...
3,12363093.0,11481705.0,https://www.policearrests.com/florida-arrest-r...,"<!DOCTYPE HTML>\n<html lang=""en"">\n<head>\n\t<...",Potentially unlawful activity,url209,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the information on this website is taken from ...
4,12376392.0,11488536.0,https://www.bustedmugshots.com/tennessee/memph...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,Potentially unlawful activity,url628,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,name clayton thomas location memphis tennessee...


In [3]:
def parse_html_text(html):
    if pd.isna(html) or html == '':
        return ''
    soup = BeautifulSoup(str(html), 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text_blocks = soup.select('h1, h2, h3, h4, h5, h6, p')
    text = ' '.join(block.get_text(' ', strip=True) for block in text_blocks) if text_blocks else soup.get_text(' ', strip=True)
    text = re.sub(r'https?://\S+|www\.\S+|\S+@\S+', ' ', text)
    text = text.replace("'", '')
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

claims = claims.copy()
if 'text_clean' not in claims.columns or claims['text_clean'].isna().all():
    claims['text_clean'] = claims['text_tmp'].map(parse_html_text)
else:
    claims['text_clean'] = claims['text_clean'].fillna('').map(lambda x: parse_html_text(x) if x else '')

claims[['.id', 'mclass', 'text_clean']].head()


,.id,mclass,text_clean
0,url2183,Possible Fatality,national obituary search click on the item you...
1,url2136,Potentially unlawful activity,the following official arrest record for jonat...
2,url1586,N/A: No relevant content.,did someone you know get arrested in miami dad...
3,url209,Potentially unlawful activity,the information on this website is taken from ...
4,url628,Potentially unlawful activity,name clayton thomas location memphis tennessee...


In [4]:
X = claims['text_clean'].fillna('')
y = claims['mclass']
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

candidate_models = {
    'logreg': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced'))
    ]),
    'linearsvc': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ('model', LinearSVC(class_weight='balanced'))
    ]),
    'multinomial_nb': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ('model', MultinomialNB())
    ]),
}

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    f1 = f1_score(y_val, pred, average='macro')
    print(f'{name}: accuracy={acc:.4f}, macro_f1={f1:.4f}')


logreg: accuracy=0.7850, macro_f1=0.7290
linearsvc: accuracy=0.8131, macro_f1=0.7376
multinomial_nb: accuracy=0.7126, macro_f1=0.5416


## Interpretation of results

The best-performing multiclass model was a TF-IDF + Linear SVC pipeline, which achieved about 81.3% validation accuracy and a macro-F1 of 0.74. This suggests that the page text contains clear signal for separating the claim categories, especially for the larger classes such as 'Possible Fatality' and 'Potentially unlawful activity.' The weaker performance for 'Other claim content' is consistent with class imbalance: there were fewer pages in that class, so the model had less evidence to learn from.

Compared with the logistic regression and Naive Bayes baselines, the SVM gave the best balance of precision and recall, so it is the most defensible final multiclass model for this task. For the binary relevant-vs.-nonrelevant prediction, the logistic regression model also performed well at about 80.6% validation accuracy, which indicates that the text is strongly informative even after collapsing the labels to two groups.


In [5]:
# Final multiclass model: an SVM trained on TF-IDF features
final_model = candidate_models['linearsvc']
final_model.fit(X_train, y_train)
val_pred = final_model.predict(X_val)
print(f'validation accuracy: {accuracy_score(y_val, val_pred):.4f}')
print(f"validation macro F1: {f1_score(y_val, val_pred, average='macro'):.4f}")
print(classification_report(y_val, val_pred))


validation accuracy: 0.8131
validation macro F1: 0.7376
                               precision    recall  f1-score   support

    N/A: No relevant content.       0.81      0.82      0.82       203
          Other claim content       0.42      0.40      0.41        20
            Physical Activity       0.76      0.60      0.67        47
            Possible Fatality       0.85      0.91      0.88        88
Potentially unlawful activity       0.90      0.93      0.92        70

                     accuracy                           0.81       428
                    macro avg       0.75      0.73      0.74       428
                 weighted avg       0.81      0.81      0.81       428



In [6]:
# Build the full prediction frame expected by the project definition
claims_test = pyreadr.read_r(str(base / 'data' / 'claims-test.RData'))['claims_test']
claims_test['text_clean'] = claims_test['text_tmp'].fillna('').map(parse_html_text)

binary_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced'))
])
binary_target = claims['bclass']
Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    X, binary_target, test_size=0.2, random_state=42, stratify=binary_target
)
binary_model.fit(Xb_train, yb_train)
print(f'binary validation accuracy: {accuracy_score(yb_val, binary_model.predict(Xb_val)):.4f}')

pred_df = claims_test[['.id']].copy()
pred_df['bclass.pred'] = binary_model.predict(claims_test['text_clean'])
pred_df['mclass.pred'] = final_model.predict(claims_test['text_clean'])
pred_df.head()


C:\Users\leelz\AppData\Local\Temp\ipykernel_36504\3462920022.py:4: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(str(html), 'html.parser')


binary validation accuracy: 0.8061


,.id,bclass.pred,mclass.pred
0,url3,Relevant claim content,Potentially unlawful activity
1,url6,N/A: No relevant content.,N/A: No relevant content.
2,url8,N/A: No relevant content.,N/A: No relevant content.
3,url9,N/A: No relevant content.,N/A: No relevant content.
4,url11,N/A: No relevant content.,N/A: No relevant content.


## Final export

This final block refits the selected binary and multiclass models on the full labeled dataset and saves the generated predictions, together with deployable model artifacts, to the `results/` folder.


In [ ]:
import pickle
import re
from pathlib import Path

import pandas as pd
import pyreadr
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

base = Path(r'C:\Users\leelz\OneDrive\UCSB\Year 4\Quarter 1 (Fall)\PSTAT 197A\module2-claims-team3-1')
results_dir = base / 'results'
results_dir.mkdir(exist_ok=True, parents=True)

def parse_html_text(html):
    if pd.isna(html) or html == '':
        return ''
    soup = BeautifulSoup(str(html), 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text_blocks = soup.select('h1, h2, h3, h4, h5, h6, p')
    text = ' '.join(block.get_text(' ', strip=True) for block in text_blocks) if text_blocks else soup.get_text(' ', strip=True)
    text = re.sub(r'https?://\S+|www\.\S+|\S+@\S+', ' ', text)
    text = text.replace("'", '')
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

claims = pyreadr.read_r(str(base / 'data' / 'claims-clean-example.RData'))['claims_clean']
claims = claims.copy()
if 'text_clean' not in claims.columns or claims['text_clean'].isna().all():
    claims['text_clean'] = claims['text_tmp'].map(parse_html_text)
else:
    claims['text_clean'] = claims['text_clean'].fillna('').map(lambda x: parse_html_text(x) if x else '')

binary_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced'))
])
multiclass_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('model', LinearSVC(class_weight='balanced'))
])

binary_model.fit(claims['text_clean'].fillna(''), claims['bclass'])
multiclass_model.fit(claims['text_clean'].fillna(''), claims['mclass'])

with open(results_dir / 'claims_binary_model.pkl', 'wb') as f:
    pickle.dump(binary_model, f)
with open(results_dir / 'claims_multiclass_model.pkl', 'wb') as f:
    pickle.dump(multiclass_model, f)

claims_test = pyreadr.read_r(str(base / 'data' / 'claims-test.RData'))['claims_test']
claims_test = claims_test.copy()
claims_test['text_clean'] = claims_test['text_tmp'].fillna('').map(parse_html_text)

pred_df = claims_test[['.id']].copy()
pred_df['bclass.pred'] = binary_model.predict(claims_test['text_clean'])
pred_df['mclass.pred'] = multiclass_model.predict(claims_test['text_clean'])

pyreadr.write_rdata(str(results_dir / 'preds-group3.RData'), pred_df, df_name='pred_df')
print('Saved binary model to', results_dir / 'claims_binary_model.pkl')
print('Saved multiclass model to', results_dir / 'claims_multiclass_model.pkl')
print('Saved predictions to', results_dir / 'preds-group3.RData')
pred_df.head()
